# RAG completo con Elasticsearch y LangChain 1.x

Este notebook practica un flujo completo de **Retrieval Augmented Generation (RAG)** usando un PDF sobre redes neuronales como fuente documental.

La práctica está dividida en tres partes:

1. **Ingesta de datos:** carga de PDF, fragmentación con `RecursiveCharacterTextSplitter`, embeddings y almacenamiento en Elasticsearch.
2. **Agente básico con RAG:** creación de un agente en LangChain 1.x usando una herramienta propia que envuelve el `vector_store` y permite filtros por metadata.
3. **Búsquedas avanzadas:** filtros por metadata, filtros por varios campos, BM25, búsqueda híbrida BM25 + vector, búsqueda híbrida + metadata, reranking y respuestas con sustento de documento y página.

> Importante: el objetivo didáctico es mostrar cómo la metadata permite pasar de una búsqueda semántica simple a una recuperación más controlada, explicable y auditable.

# Instalación de dependencias


In [1]:
%pip install -q "langchain>=1.0" langchain-google-genai langchain-elasticsearch langchain-community langchain-text-splitters pypdf elasticsearch pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.8/952.8 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


# 1. Configuración de credenciales y variables


- `/content/api_key.txt` para `GEMINI_API_KEY`
- `/content/elasticstore_urp.txt` para la contraseña de Elasticsearch

También puedes usar variables de entorno.

In [2]:
import os
import getpass
from pathlib import Path


def read_secret_file(path: str) -> str | None:
    file_path = Path(path)
    if file_path.exists():
        return file_path.read_text(encoding="utf-8").strip()
    return None

# Gemini
gemini_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY") or read_secret_file("/content/api_key.txt")
if not gemini_key:
    gemini_key = getpass.getpass("Ingresa tu GEMINI_API_KEY: ")
os.environ["GEMINI_API_KEY"] = gemini_key
os.environ["GOOGLE_API_KEY"] = gemini_key

# Elasticsearch
ES_URL = os.getenv("ES_URL", "http://104.198.172.31:9200")
ES_USER = os.getenv("ES_USER", "elastic")
ES_PASSWORD = os.getenv("ES_PASSWORD") or read_secret_file("/content/elasticstore_urp.txt")
if not ES_PASSWORD:
    ES_PASSWORD = getpass.getpass("Ingresa la contraseña de Elasticsearch: ")

INDEX_NAME = os.getenv("ES_INDEX", "rag_hiraoka_legales_envio")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "gemini-embedding-2")
CHAT_MODEL = os.getenv("CHAT_MODEL", "google_genai:gemini-2.5-flash")

print("Configuración cargada")
print("ES_URL:", ES_URL)
print("INDEX_NAME:", INDEX_NAME)
print("EMBEDDING_MODEL:", EMBEDDING_MODEL)
print("CHAT_MODEL:", CHAT_MODEL)

Configuración cargada
ES_URL: http://104.198.172.31:9200
INDEX_NAME: rag_hiraoka_legales_envio
EMBEDDING_MODEL: gemini-embedding-2
CHAT_MODEL: google_genai:gemini-2.5-flash


# Parte 1 — Ingesta de datos

En esta parte cargaremos el PDF, construiremos metadata útil para búsquedas filtradas y guardaremos los chunks en Elasticsearch.

La metadata se construye considerando que el PDF trata sobre **redes neuronales**, con secciones como introducción, reseña histórica, generalidades, elementos básicos, aprendizaje, topologías, aplicaciones y software comercial.

## 1.1 Cargar el PDF en Colab

El notebook espera el archivo `matich-redesneuronales.pdf`. Si no existe en `/content`, se abrirá un uploader para cargarlo manualmente.

In [3]:
PDF_PATH = Path("/content/Legales_Envios.pdf")

if not PDF_PATH.exists():
    try:
        from google.colab import files
        print("Sube el archivo Terminos_Condiciones.pdf")
        uploaded = files.upload()
        if uploaded:
            first_name = next(iter(uploaded.keys()))
            Path(first_name).rename(PDF_PATH)
            print(f"PDF cargado como: {PDF_PATH}")
    except Exception as e:
        raise FileNotFoundError(
            "No se encontró /content/Legales_Envios.pdf. "
            "Sube el PDF manualmente o cambia PDF_PATH."
        ) from e

print("PDF listo:", PDF_PATH)

PDF listo: /content/Legales_Envios.pdf


## 1.2 Loader para PDF

Usaremos `PyPDFLoader` para extraer el texto por página. Cada página se convierte en un `Document` de LangChain.

In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()

print(f"Páginas cargadas: {len(pages)}")
print("Metadata base:", pages[0].metadata)

/tmp/ipykernel_2773/3086807236.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Páginas cargadas: 3
Metadata base: {'producer': 'Skia/PDF m149', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36 Edg/149.0.0.0', 'creationdate': '2026-06-18T16:05:01+00:00', 'title': 'Legales Envios', 'moddate': '2026-06-18T16:05:01+00:00', 'source': '/content/Legales_Envios.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}


## 1.3 Enriquecimiento de metadata

La metadata permitirá filtrar por:

- `documento`
- `page_number`
- `section_slug`
- `section_title`
- `temas`
- `tipo_contenido`
- `dominio`
- `autor`
- `anio`

Esto es importante porque luego la herramienta del agente podrá buscar por tema, página, sección o tipo de contenido.

### Paso 1 - Crear secciones reales del documento

In [5]:
import re
import unicodedata
from dataclasses import dataclass
from typing import Optional

DOCUMENT_TITLE = "Legales Envios"
DOCUMENT_AUTHOR = "Hiraoka"
DOCUMENT_YEAR = 2026
DOCUMENT_DOMAIN = "Legales Envios"


def normalize_text(text: str) -> str:
    return (
        unicodedata.normalize("NFKD", text.lower())
        .encode("ascii", "ignore")
        .decode("ascii")
    )


@dataclass
class Section:
    slug: str
    title: str
    parent: Optional[str]
    page_start: int
    page_end: int
    text: str

SECTION_TITLES = [

    # Documento
    {   "slug": "envios_hoy", "title": "1. ENVÍO HOY"},

    # I
    {
        "slug": "envio_regular", "title": "2. ENVIO REGULAR (24hrs a más)"
    },

    {
        "slug": "entrega_tienda",
        "title": "3. ENTREGA EN TIENDA"
    },

    {
        "slug": "same_day", "title": "Same Day"
    },

    {
        "slug": "envio_regular", "title": "Envío Regular:"
    },
    {
        "slug": "consideración",
        "title": "consideración",
        "parent": "envio_regular"
    }
]

### Paso 2 - Encontrar dónde empieza cada sección

In [6]:
def find_section_starts(pages):
    full_text = ""
    char_to_page = []

    for idx, page in enumerate(pages):
        start = len(full_text)
        full_text += page.page_content + "\n\n"
        end = len(full_text)
        char_to_page.append((start, end, idx + 1))

    def get_page_number(char_idx):
        for start, end, p_num in char_to_page:
            if start <= char_idx < end:
                return p_num
        return len(pages)

    norm_full = normalize_text(full_text)
    detected = []

    for sec in SECTION_TITLES:
        norm_title = normalize_text(sec["title"])
        pos = norm_full.find(norm_title)
        if pos != -1:
            detected.append({
                "slug": sec["slug"],
                "title": sec["title"],
                "parent": sec.get("parent"),
                "char_start": pos,
                "page": get_page_number(pos)
            })

    detected.sort(key=lambda x: x["char_start"])
    return detected, full_text, get_page_number

### Paso 3 - Construir objetos Section

In [7]:
def build_sections(pages):
    starts, full_text, get_page_number = find_section_starts(pages)
    sections = []

    for i, section in enumerate(starts):
        start_pos = section["char_start"]
        end_pos = starts[i+1]["char_start"] if i < len(starts)-1 else len(full_text)

        # Extraer el texto exacto entre esta sección y la siguiente
        section_text = full_text[start_pos:end_pos].strip()
        page_start = section["page"]
        page_end = get_page_number(end_pos - 1)

        sections.append(
            Section(
                slug=section["slug"],
                title=section["title"],
                parent=section["parent"],
                page_start=page_start,
                page_end=page_end,
                text=section_text
            )
        )
    return sections

### Ejecutar

In [8]:
sections = build_sections(pages)

print(f"Secciones encontradas: {len(sections)}")

for s in sections:
    print(
        s.slug,
        s.page_start,
        s.page_end
    )

Secciones encontradas: 6
envios_hoy 1 1
envio_regular 1 1
entrega_tienda 1 2
same_day 2 2
envio_regular 2 3
consideración 3 3


## 1.4 Fragmentación con RecursiveCharacterTextSplitter

Fragmentaremos por página para preservar la trazabilidad. Cada chunk hereda la metadata de su página original.

In [9]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1300,
    chunk_overlap=250
)

### Convertimos las secciones a documentos:

In [10]:
section_docs = []

# Mapeamos los títulos de secciones por su slug para resolver la jerarquía
sections_by_slug = {s["slug"]: s for s in SECTION_TITLES}

for section in sections:
    metadata = {
        "documento": DOCUMENT_TITLE,
        "autor": DOCUMENT_AUTHOR,
        "anio": DOCUMENT_YEAR,
        "dominio": DOCUMENT_DOMAIN,
        "page_start": section.page_start,
        "page_end": section.page_end
    }

    # Determinar jerarquía
    parent_slug = section.parent
    if not parent_slug:
        # Es sección principal (Main)
        metadata["section_slug"] = section.slug
        metadata["section_title"] = section.title
    else:
        grandparent_slug = sections_by_slug[parent_slug].get("parent")
        if not grandparent_slug:
            # Es subsección (Sub)
            metadata["section_slug"] = parent_slug
            metadata["section_title"] = sections_by_slug[parent_slug]["title"]
            metadata["subsection_slug"] = section.slug
            metadata["subsection_title"] = section.title
        else:
            # Es sub-subsección (Sub-sub)
            metadata["section_slug"] = grandparent_slug
            metadata["section_title"] = sections_by_slug[grandparent_slug]["title"]
            metadata["subsection_slug"] = parent_slug
            metadata["subsection_title"] = sections_by_slug[parent_slug]["title"]

            metadata["sub_subsection_title"] = section.title

    doc = Document(
        page_content=section.text,
        metadata=metadata
    )
    section_docs.append(doc)

In [11]:
splits = splitter.split_documents(
    section_docs
)

In [12]:
print(splits[0].page_content[:900])

1. ENVÍO HOY
La entrega del producto se realizará observando lo siguiente:
Si la compra de los productos/pedido se realiza entre la 07:00 a las 13:00 horas, los
productos/pedidos comprados fuera de este horario, no tendrán la opción de seleccionar
este método de despacho.
Se recibirá el producto durante el mismo día en que realizó la compra, el rango de
entrega del pedido al cliente será de las 15:00 a las 20:00 horas.
Aplica para todas las tallas de productos (XS, S, M, SL,)
El servicio está sujeto a la disponibilidad de despacho por día, cuando la capacidad de
pedidos por día es completada (Máximo de 50 pedidos diarios - Sujeto a variación) o
cuando se encuentre fuera del rango horario, el servicio se dejará de ofrecer,
independientemente de cuál ocurra primero.
No aplica para métodos de Pago Diferido (Pago Efectivo y similares)
Durante las festividades de Navidad y Año Nuevo, el horar


### Paso 5 - Metadata a nivel chunk

In [13]:
for idx, chunk in enumerate(splits):
    specific_slug = (
        chunk.metadata.get("subsection_slug")
        or chunk.metadata.get("section_slug")
    )
    chunk.metadata["chunk_id"] = f"{specific_slug}_{idx:04d}"
    chunk.metadata["chunk_number"] = idx

In [14]:
print(splits[2].metadata)

{'documento': 'Legales Envios', 'autor': 'Hiraoka', 'anio': 2026, 'dominio': 'Legales Envios', 'page_start': 1, 'page_end': 2, 'section_slug': 'entrega_tienda', 'section_title': '3. ENTREGA EN TIENDA', 'chunk_id': 'entrega_tienda_0002', 'chunk_number': 2}


## 1.5 Resumen de metadata generada

Antes de indexar, revisamos la distribución de chunks por sección y tipo de contenido.

In [15]:
import pandas as pd

metadata_df = pd.DataFrame([doc.metadata for doc in splits])

resumen_secciones = (
    metadata_df.groupby(["section_title", "page_start"])
    .size()
    .reset_index(name="chunks")
    .sort_values("chunks", ascending=False)
)

resumen_tipo = (
    metadata_df.groupby("section_title")
    .size()
    .reset_index(name="chunks")
    .sort_values("chunks", ascending=False)
)

print("Chunks por sección")
display(resumen_secciones)
print("Chunks por tipo de contenido")
display(resumen_tipo)

Chunks por sección


,section_title,page_start,chunks
2,3. ENTREGA EN TIENDA,1,4
4,Envío Regular:,3,3
3,Envío Regular:,2,2
0,1. ENVÍO HOY,1,1
1,2. ENVIO REGULAR (24hrs a más),1,1
5,Same Day,2,1


Chunks por tipo de contenido


,section_title,chunks
3,Envío Regular:,5
2,3. ENTREGA EN TIENDA,4
0,1. ENVÍO HOY,1
1,2. ENVIO REGULAR (24hrs a más),1
4,Same Day,1


## 1.6 Embeddings y conexión a Elasticsearch

Conectamos a `ElasticsearchStore` y guardaremos los chunks con IDs estables basados en documento, página y número de chunk.

In [16]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore

embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = ElasticsearchStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    es_url=ES_URL,
    es_user=ES_USER,
    es_password=ES_PASSWORD,
)

# Cambia a False si quieres reutilizar el índice ya creado.
RECREATE_INDEX = True

if RECREATE_INDEX and vector_store.client.indices.exists(index=INDEX_NAME):
    vector_store.client.indices.delete(index=INDEX_NAME)
    print(f"Índice eliminado: {INDEX_NAME}")

ids = [doc.metadata["chunk_id"] for doc in splits]

inserted_ids = vector_store.add_documents(
    documents=splits,
    ids=ids,
    bulk_kwargs={"chunk_size": 100},
)

vector_store.client.indices.refresh(index=INDEX_NAME)
print(f"Chunks indexados: {len(inserted_ids)}")

Chunks indexados: 12


## 1.7 Validación rápida de recuperación vectorial

Probamos una búsqueda semántica simple antes de construir el agente.

In [17]:
from collections import Counter


def score_label(score):

    if score is None:
        return ""

    if score >= 0.90:
        return "🟢 Excelente"

    if score >= 0.80:
        return "🟡 Bueno"

    return "🔴 Débil"


def print_retrieval_results(results, max_chars=600):

    print("\n" + "=" * 120)
    print("RESULTADOS DE RECUPERACIÓN")
    print("=" * 120)

    section_counter = Counter()

    for i, item in enumerate(results, start=1):

        # Compatible con similarity_search()
        # y similarity_search_with_score()

        if isinstance(item, tuple):
            doc, score = item
        else:
            doc, score = item, None

        meta = doc.metadata

        section_slug = meta.get("section_slug", "N/A")
        section_counter[section_slug] += 1

        print("\n" + "=" * 120)

        header = f"Resultado {i}"

        if score is not None:
            header += (
                f" | score={score:.4f}"
                f" ({score_label(score)})"
            )

        print(header)

        print("-" * 120)

        print(
            f"Chunk ID      : "
            f"{meta.get('chunk_id')}"
        )

        print(
            f"Documento     : "
            f"{meta.get('documento')}"
        )

        print(
            f"Sección       : "
            f"{meta.get('section_title')} "
            f"({meta.get('section_slug')})"
        )

        print(
            f"Páginas       : "
            f"{meta.get('page_start')} "
            f"- "
            f"{meta.get('page_end')}"
        )

        print(
            f"Autor         : "
            f"{meta.get('autor')}"
        )

        print(
            f"Año           : "
            f"{meta.get('anio')}"
        )

        print(
            f"Dominio       : "
            f"{meta.get('dominio')}"
        )

        print(
            f"Categoría SAC : "
            f"{meta.get('sac_category', 'N/A')}"
        )

        print(
            f"Tipo Cláusula : "
            f"{meta.get('clause_type', 'N/A')}"
        )

        print(
            f"Nivel Riesgo  : "
            f"{meta.get('risk_level', 'N/A')}"
        )

        entities = meta.get("entities", [])

        print(
            f"Entidades     : "
            f"{', '.join(entities) if entities else 'N/A'}"
        )

        print("-" * 120)

        preview = (
            doc.page_content[:max_chars]
            .replace("\n", " ")
            .strip()
        )

        print(preview)

    print("\n" + "=" * 120)
    print("DISTRIBUCIÓN POR SECCIONES")
    print("=" * 120)

    for section, qty in section_counter.items():
        print(f"{section:<30} {qty}")

In [18]:
query = "¿Puedo mandar a otra persona a recoger?"

results = vector_store.similarity_search_with_score(
    query,
    k=5
)

print_retrieval_results(results)


RESULTADOS DE RECUPERACIÓN

Resultado 1 | score=0.8491 (🟡 Bueno)
------------------------------------------------------------------------------------------------------------------------
Chunk ID      : entrega_tienda_0003
Documento     : Legales Envios
Sección       : 3. ENTREGA EN TIENDA (entrega_tienda)
Páginas       : 1 - 2
Autor         : Hiraoka
Año           : 2026
Dominio       : Legales Envios
Categoría SAC : N/A
Tipo Cláusula : N/A
Nivel Riesgo  : N/A
Entidades     : N/A
------------------------------------------------------------------------------------------------------------------------
En caso el cliente optase por recoger el producto personalmente, deberá presentarse con su documento oficial de identidad original y vigente al área de “Paquetes” de la tienda seleccionada. En caso el cliente decida realizar el recojo del producto por medio de otra persona natural, el cliente deberá indicar el nombre completo y documento oficial de identidad de dicha persona natural en los 

# Parte 2 — Agente básico con RAG mediante herramienta propia

En esta parte construiremos un agente con `create_agent`, pero la recuperación no se expondrá con `retriever.as_tool()` ni con `chain.as_tool()`.

Crearemos una herramienta propia con `@tool` que envuelve el `vector_store` y permite filtros por metadata. Este enfoque permite controlar:

- `section_slug`
- rango de páginas
- `tipo_contenido`
- `temas`
- cantidad de resultados
- formato de salida con fuentes

## 2.1 Constructor de filtros por metadata

Elasticsearch permite filtrar usando expresiones como `term`, `terms` y `range`. La metadata quedó guardada dentro del objeto `metadata`, por eso los filtros usan campos como `metadata.section_slug.keyword` o `metadata.page_number`.

In [19]:
from typing import Optional


def build_metadata_filter(
    section_slug: Optional[str] = None,
    documento: Optional[str] = None,
) -> list[dict]:
    """
    Construye filtros Elasticsearch para búsqueda híbrida RAG.

    Metadata soportada actualmente:
    - documento
    - section_slug (soporta buscar en cualquier nivel de la jerarquía: section, subsection, sub_subsection)
    """

    filters = []

    if documento:
        filters.append({
            "term": {
                "metadata.documento.keyword": documento
            }
        })

    if section_slug:
        filters.append({
            "bool": {
                "should": [
                    {"term": {"metadata.section_slug.keyword": section_slug}},
                    {"term": {"metadata.subsection_slug.keyword": section_slug}}
                ],
                "minimum_should_match": 1
            }
        })

    return filters

In [20]:
filters = build_metadata_filter(
    documento="Legales_Envios",
    section_slug="envios_hoy"
)

print(filters)

[{'term': {'metadata.documento.keyword': 'Legales_Envios'}}, {'bool': {'should': [{'term': {'metadata.section_slug.keyword': 'envios_hoy'}}, {'term': {'metadata.subsection_slug.keyword': 'envios_hoy'}}], 'minimum_should_match': 1}}]


## 2.2 Herramienta RAG personalizada

La herramienta recibe parámetros semánticos y estructurados. Internamente construye filtros de Elasticsearch y ejecuta la búsqueda sobre el vector store.

In [21]:
from typing import Optional

from pydantic import BaseModel, Field


class BuscarTerminosHiraokaInput(BaseModel):

    consulta: str = Field(
        description="Pregunta relacionada con legales de envios de Hiraoka."
    )

    k: int = Field(
        default=5,
        ge=1,
        le=10,
        description="Cantidad máxima de fragmentos a recuperar."
    )

    section_slug: Optional[str] = Field(
        default=None,
        description="""
        Sección específica del documento.

        Ejemplos:
        - Envios
        - Envios a provincia
        """
    )

In [22]:
SECTION_HINTS = {

    # ENVIO HOY
    "envio hoy": "envios_hoy",
    "envío hoy": "envios_hoy",
    "hoy mismo": "envios_hoy",
    "mismo dia": "envios_hoy",
    "mismo día": "envios_hoy",
    "entrega hoy": "envios_hoy",
    "llega hoy": "envios_hoy",

    # ENVIO REGULAR
    "envio regular": "envio_regular",
    "envío regular": "envio_regular",
    "24 horas": "envio_regular",
    "24hrs": "envio_regular",
    "fecha entrega": "envio_regular",
    "programar entrega": "envio_regular",
    "turno entrega": "envio_regular",
    "primer turno": "envio_regular",
    "segundo turno": "envio_regular",

    # ENTREGA EN TIENDA
    "recojo": "entrega_tienda",
    "retiro": "entrega_tienda",
    "recojo tienda": "entrega_tienda",
    "retiro tienda": "entrega_tienda",
    "entrega tienda": "entrega_tienda",
    "recoger producto": "entrega_tienda",

    # DOCUMENTOS
    "dni": "entrega_tienda",
    "documento identidad": "entrega_tienda",
    "tercero": "entrega_tienda",
    "otra persona": "entrega_tienda",
    "apoderado": "entrega_tienda",

    # TIEMPOS RECOJO
    "cuantos dias recojo": "entrega_tienda",
    "15 dias": "entrega_tienda",
    "15 días": "entrega_tienda",
    "penalidad": "entrega_tienda",
    "penalidades": "entrega_tienda",
    "s/50": "entrega_tienda",

    # SAME DAY
    "same day": "same_day",
    "entrega mismo dia": "same_day",
    "entrega mismo día": "same_day",

    # COBERTURA
    "cobertura": "same_day",
    "distritos": "same_day",
    "distrito": "same_day",
    "lima": "same_day",
    "callao": "same_day",

    # PROVINCIAS
    "arequipa": "envio_regular",
    "cusco": "envio_regular",
    "piura": "envio_regular",
    "trujillo": "envio_regular",
    "ica": "envio_regular",
    "huancayo": "envio_regular",
    "provincia": "envio_regular",
    "provincias": "envio_regular",

    # DELIVERY
    "delivery": "envio_regular",
    "envio": "envio_regular",
    "envío": "envio_regular",
    "despacho": "envio_regular",

    # INSTALACION
    "instalacion": "consideracion",
    "instalación": "consideracion",
    "instalar": "consideracion",

    # ESCALERAS / ASCENSOR
    "escalera": "consideracion",
    "ascensor": "consideracion",
    "subir piso": "consideracion",
    "puerta": "consideracion",
    "ventana": "consideracion",

    # PROPINAS
    "propina": "consideracion",
    "propinas": "consideracion",

    # REENTREGA
    "segunda entrega": "consideracion",
    "no habia nadie": "consideracion",
    "no había nadie": "consideracion",
    "no pudieron entregar": "consideracion",
}

In [23]:
def format_results_for_agent(
    results,
    max_chars: int = 1200
):

    if not results:
        return (
            "No se encontró información relevante "
            "Cambios y Devoluciones."
        )

    blocks = []

    for i, (doc, score) in enumerate(results, start=1):

        meta = doc.metadata

        content = (
            doc.page_content[:max_chars]
            .replace("\n", " ")
            .strip()
        )

        # Construir título jerárquico de la sección
        sec_title = meta.get('section_title', '')
        if meta.get('subsection_title'):
            sec_title += f" -> {meta.get('subsection_title')}"
        if meta.get('sub_subsection_title'):
            sec_title += f" -> {meta.get('sub_subsection_title')}"

        blocks.append(
            f"""
[Fuente {i}]

Documento: {meta.get('documento')}

Sección: {sec_title}

Slug: {meta.get('subsection_slug') or meta.get('section_slug')}

Páginas: {meta.get('page_start')} - {meta.get('page_end')}

Chunk ID: {meta.get('chunk_id')}

Score: {score:.4f}

Contenido:
{content}
"""
        )

    return "\n\n".join(blocks)

### Herramienta principal

In [24]:
from langchain.tools import tool


@tool(args_schema=BuscarTerminosHiraokaInput)
def buscar_terminos_hiraoka(
    consulta: str,
    k: int = 5,
    section_slug: Optional[str] = None,
) -> str:
    """
    Busca información dentro de los
    Cambios y Devoluciones de Hiraoka
    utilizando Elasticsearch.
    """

    # Detección automática de sección

    if not section_slug:

        query_lower = consulta.lower()

        for keyword, detected_section in SECTION_HINTS.items():

            if keyword in query_lower:

                section_slug = detected_section
                break

    filters = build_metadata_filter(
        section_slug=section_slug,
        documento= DOCUMENT_TITLE
    )

    results = vector_store.similarity_search_with_score(
        query=consulta,
        k=k,
        filter=filters or None,
    )

    results = [
        r for r in results
        if r[1] >= 0.65
    ]

    results = sorted(
        results,
        key=lambda x: x[1],
        reverse=True
    )

    return format_results_for_agent(results)

In [25]:
print(
    buscar_terminos_hiraoka.invoke(
        {
            "consulta": "¿Llegan a Cusco?"
        }
    )
)


[Fuente 1]

Documento: Legales Envios

Sección: Envío Regular:

Slug: envio_regular

Páginas: 2 - 3

Chunk ID: envio_regular_0008

Score: 0.8528

Contenido:
Playas Sur (Frecuencia de entrega semanal Sábados): Lurín, Pachacamác, Pucusana, Punta Hermosa, Punta Negra, San Bartolo, Santa Maria del Mar, Asia, Chilca, Mala, San Antonio. Provincias (frecuencia de entrega diaria):  Arequipa, llegando a los distritos de Arequipa, Cayma, Cerro Colorado, Jacobo Hunter, José Luis Bustamente y Rivero, Mariano Melgar, Miraflores, Paucarpata, Sachaca, Socabaya, Tiabaya, Yanahuara; en Lambayeque a las distritos de Chiclayo y La Victoria; en Piura a los distritos de Piura, Castilla, 26 de Octubre, Sullana y Mancora; en La Libertad a los distritos de Trujillo, Victor Larco Herrera, Pacasmayo y Chepén; y en Ica, a los distritos de Ica, Los Aquijes, Pachacutec, Parcona, Pueblo Nuevo (Ica), San Juan Bautista, Santiago, Chincha Alta, Chincha Baja, Alto Laran, El Carmen, Grocio Prado, Pueblo Nuevo (Chincha)

In [ ]:
print(splits[0].metadata)

{'documento': 'Cambios y Devoluciones', 'autor': 'Hiraoka', 'anio': 2026, 'dominio': 'Cambios y Devoluciones', 'page_start': 1, 'page_end': 1, 'section_slug': 'politica_cambios_devoluciones_garantias', 'section_title': 'POLÍTICA DE CAMBIOS, DEVOLUCIONES Y GARANTIAS', 'chunk_id': 'politica_cambios_devoluciones_garantias_0000', 'chunk_number': 0}


## 2.3 Prueba directa de la herramienta

Antes de conectar la herramienta al agente, conviene probarla de forma aislada.

In [26]:
test_questions = [

    "¿Hacen delivery a Huacho?",
    "¿Puedo mandar a otra persona a recoger?",
    "¿Cuánto cobran de penalidad?",
    "¿El delivery incluye instalación?"

]

for question in test_questions:

    print("\n\n")
    print("#" * 120)
    print("PREGUNTA")
    print(question)
    print("#" * 120)

    results = vector_store.similarity_search_with_score(
        question,
        k=3
    )

    print_retrieval_results(
        results,
        max_chars=400
    )




########################################################################################################################
PREGUNTA
¿Hacen delivery a Huacho?
########################################################################################################################

RESULTADOS DE RECUPERACIÓN

Resultado 1 | score=0.8812 (🟡 Bueno)
------------------------------------------------------------------------------------------------------------------------
Chunk ID      : envio_regular_0008
Documento     : Legales Envios
Sección       : Envío Regular: (envio_regular)
Páginas       : 2 - 3
Autor         : Hiraoka
Año           : 2026
Dominio       : Legales Envios
Categoría SAC : N/A
Tipo Cláusula : N/A
Nivel Riesgo  : N/A
Entidades     : N/A
------------------------------------------------------------------------------------------------------------------------
Playas Sur (Frecuencia de entrega semanal Sábados): Lurín, Pachacamác, Pucusana, Punta Hermosa, Punta Negra, San Bartolo,

## 2.4 Crear agente básico con LangChain 1.x

El agente tendrá una sola herramienta de recuperación. La instrucción principal obliga a responder con sustento: documento, sección y página.

In [27]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(
    CHAT_MODEL,
    temperature=0
)

SYSTEM_PROMPT = """
Eres un asistente especializado en las Políticas de Envío y Entrega de Hiraoka.

Tu objetivo es responder consultas relacionadas con:

- Envío Hoy
- Envío Regular
- Same Day
- Delivery
- Cobertura de distritos
- Cobertura de provincias
- Recojo en tienda
- Entrega en tienda
- Costos de envío
- Tiempos de entrega
- Reprogramaciones
- Restricciones de entrega
- Validación de identidad
- Terceros autorizados
- Penalidades de recojo
- Consideraciones de despacho

INFORMACIÓN FIJA

Datos de contacto:

- Teléfono: (01) 680-3800
- WhatsApp: 969872372
- Correo: servicioalcliente@hiraoka.com.pe

INSTRUCCIONES

1. No inventes información.

2. Utiliza la herramienta buscar_terminos_hiraoka para responder consultas relacionadas con las políticas de envío y entrega.

3. Prioriza siempre la información recuperada desde la herramienta.

4. Si existen varias modalidades de entrega aplicables, explica las diferencias.

Por ejemplo:

- Envío Hoy
- Same Day
- Envío Regular
- Entrega en Tienda

5. Responde siempre en español.

6. Si la consulta está fuera del alcance de las políticas de envío y entrega, indícalo claramente.

7. Si la consulta es únicamente sobre:

- teléfono
- whatsapp
- correo

NO utilices la herramienta.

8. Utiliza la herramienta para consultas relacionadas con:

- delivery
- envío
- despacho
- cobertura
- distrito
- provincia
- same day
- envío hoy
- envío regular
- recojo
- retiro en tienda
- entrega en tienda
- tiempo de entrega
- fecha de entrega
- programación
- reprogramación
- validación de identidad
- tercero autorizado
- DNI
- penalidad
- instalación
- ascensor
- escalera
- propina
- costos de envío

9. Si la herramienta no devuelve evidencia suficiente responde:

"No encontré información suficiente en las políticas de envío y entrega de Hiraoka para responder esta consulta."

FORMATO DE CITAS

Cuando uses información recuperada desde la herramienta agrega:

Fuentes:
- Documento: <documento>
- Sección: <section_title>
- Páginas: <page_start>-<page_end>

Si la respuesta proviene únicamente de la información fija, no es necesario mostrar fuentes.
"""

agent = create_agent(
    model=model,
    tools=[buscar_terminos_hiraoka],
    system_prompt=SYSTEM_PROMPT,
)

In [28]:
question = "¿Incluye instalación?"
response = agent.invoke({"messages": [{"role": "user", "content": question}]})
print(response["messages"][-1].content)

[{'type': 'text', 'text': 'No encontré información suficiente en las políticas de envío y entrega de Hiraoka para responder esta consulta.', 'extras': {'signature': 'CuABAQw51sfBym72d/yOuSyq4Qo3k52AbV1dZICGb4GGlVsNuKKVVNduAYVnWh2PRPGHhtbeM8MiRbVj8SZXOOgLxt4Uou5rIrKuKVmYJNrwltlYlX0WdRhZinVAU8BeqLtWPaptjMkgZMDUlEAoo6W5kvZ9dOBO7XzuPCgGGi7keYQLonQsN9/2RE9vj3bNqGD5ma7shnJyatbAOC9oefsybY784sXFVB/RrObhLshIYxvnT3nLUGCS1yd3fu0256/Xm/m3VMARSsB0qNEzQcIMLNbyR/cZv8PKyX9bdsvsfr0='}}]


## 2.5 Preguntas de práctica para el agente

In [29]:
preguntas_practica = [
    "¿Hacen delivery a Arequipa?",
    "¿Hacen delivery a provincias?",
    "¿Cuánto cuesta el envío?",
    "¿Qué distritos tienen cobertura Same Day?",
    "¿Cuál es la diferencia entre Envío Hoy y Same Day?",
    "¿Cuánto tiempo demora el despacho?",
    "¿Puedo cambiar la fecha de entrega?",
    "¿Qué pasa si no estoy en casa cuando llega el pedido?",
    "¿Cobran por una segunda visita?",
    "¿Puedo autorizar a otra persona para recibir el pedido?",
    "¿Qué documentos debo presentar para recoger en tienda?",
    "¿Cuánto tiempo guardan mi pedido para recojo?",
    "¿Qué pasa si no recojo mi compra dentro del plazo?",
    "¿El delivery incluye instalación del producto?",
    "¿El personal sube productos por escaleras?",
    "¿Qué sucede si el producto no entra por la puerta o ascensor?"
]

In [30]:
for pregunta in preguntas_practica[:5]:

    print("=" * 120)
    print("PREGUNTA:")
    print(pregunta)

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": pregunta
                }
            ]
        }
    )

    print("\nRESPUESTA:")
    print(response["messages"][-1].content)
    print("\n")

PREGUNTA:
¿Hacen delivery a Arequipa?

RESPUESTA:
[{'type': 'text', 'text': 'Sí, Hiraoka realiza delivery a Arequipa a través de la modalidad de Envío Regular. La frecuencia de entrega es diaria y llega a los siguientes distritos: Arequipa, Cayma, Cerro Colorado, Jacobo Hunter, José Luis Bustamente y Rivero, Mariano Melgar, Miraflores, Paucarpata, Sachaca, Socabaya, Tiabaya y Yanahuara.\n\nFuentes:\n- Documento: Legales Envios\n- Sección: Envío Regular:\n- Páginas: 2-3', 'extras': {'signature': 'Cr0EAQw51sdZ2jOB5/VyYs4oJjbx/h4WqL3Dr8SQdrISeqC+cv4w3tcNs586U0rKXRNxkm5a9sZGWmnKBguE3sp821CQLOO48KVNuwlUwtc7Vm7Vz9fRnRwzw9YczNKVljSKK0kKKp7geKvtfvHhlT0aMSG9vhn0prb5rawnLoKzsKsUbFEUorQKDS+KljcWrI9WmkBs7OZ8HY9kQ/uSjmw68/6o9QQKfKy3gjigCJT5Skki/1QYZPHsY45EtLF+tif1Q4iThcliE+46D9D52GGxF0fQLYHu35K/ogYtTFNpS2kLyxNC9C6aVQdHm2nG755dN7b5yCkZ7/Ep1CR3EQRYYndaJR5IYyCUBGw/0hU6OGTo0MGmnmOo5h30o+r55l5s/XxPMVpTNr+ygG1CkZ6NYEWcy9pOHV1Ew7j31DjMbiuZ3Uj+fHz073Enbna2ljp9OlQfBBCu3c/mlB80R4Ez9Som4IKO9J5u9Rnk7Mf4N5/kG5l

# -----------------------